In [1]:
import pandas as pd

# Jaren waarvoor we ongevallen verwerken.
# Ongevallen kunnen voor alle jaren; het verkeer (INWEVA) van 2024 en 2025 staat in aparte bestanden (komt later).
JAREN = range(2018, 2026)

# Opbouw van dit notebook:
#   1. maak_baanvakken()            -> één keer: de snelwegbaanvakken uit INWEVA (netwerk van 2023)
#   2. verwerk_jaar(jaar, baanvakken) -> per jaar: ongevallen inlezen, filteren en koppelen aan een baanvak
#   3. alle jaren onder elkaar zetten in één tabel

In [2]:
#0. data downloaden (alleen als die er nog niet is)
# download_data.py staat in dezelfde map als dit notebook. Het downloadt BRON 2018-2025 en INWEVA 2023-2025
# naar de map data/. Jaren die al aanwezig zijn worden overgeslagen; de eerste keer duurt dit even (enkele GB).
from download_data import download_alles

download_alles()

01-01-2018_31-12-2018: downloaden van https://downloads.rijkswaterstaatdata.nl/bron/01-01-2018_31-12-2018.zip
  01-01-2018_31-12-2018.zip: 227 / 227 MB
01-01-2018_31-12-2018: uitpakken
01-01-2019_31-12-2019: downloaden van https://downloads.rijkswaterstaatdata.nl/bron/01-01-2019_31-12-2019.zip
  01-01-2019_31-12-2019.zip: 227 / 227 MB
01-01-2019_31-12-2019: uitpakken
01-01-2020_31-12-2020: downloaden van https://downloads.rijkswaterstaatdata.nl/bron/01-01-2020_31-12-2020.zip
  01-01-2020_31-12-2020.zip: 223 / 223 MB
01-01-2020_31-12-2020: uitpakken
01-01-2021_31-12-2021: downloaden van https://downloads.rijkswaterstaatdata.nl/bron/01-01-2021_31-12-2021.zip
  01-01-2021_31-12-2021.zip: 295 / 295 MB
01-01-2021_31-12-2021: uitpakken
01-01-2022_31-12-2022: downloaden van https://downloads.rijkswaterstaatdata.nl/bron/01-01-2022_31-12-2022.zip
  01-01-2022_31-12-2022.zip: 298 / 298 MB
01-01-2022_31-12-2022: uitpakken
01-01-2023_31-12-2023: downloaden van https://downloads.rijkswaterstaatdata

In [3]:
#1. baanvakken maken (één keer)
# INWEVA gebruikt het wegennet van 2023. Daarom gebruiken we hier altijd wegvakken.txt van 2023,
# ook als we later ongevallen van andere jaren koppelen.

def maak_baanvakken():
    # INWEVA historie: één rij per baanvak (stuk rijbaan in één richting) met het verkeer per dag van 2018-2023
    jaarkolommen = [f"etmaal_AL_weekdag_{j}" for j in range(2018, 2024)]
    inweva = pd.read_excel("data/INWEVA/INWEVA_2023/A. Outputbestanden Excel/INWEVA_2023_historie.xlsx",
                           usecols=["VIAS_baannr", "Baantype_van", "Wegnr_van", "Ri_van", "Hm_van", "Hm_naar",
                                    "Lengte_baan_meter", "NwbId", "Traject_van", "Traject_naar"] + jaarkolommen)

    inweva_hr = inweva[inweva["Baantype_van"] == "HR"].copy()  # alleen hoofdrijbanen (geen op-/afritten)

    # wegennet 2023: welke wegvakken zijn een hoofdrijbaan van een snelweg?
    wegvakken = pd.read_csv("data/BRON/01-01-2023_31-12-2023/Netwerkgegevens/wegvakken.txt",
                            encoding="latin-1", dtype=str, usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR"])
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # hoofdrijbaan
        & (wegvakken["ROUTELTR"] == "A")     # snelweg
    ]

    #alleen snelwegen filteren
    # INWEVA bevat ook N-wegen, maar heeft geen kolom "A of N".
    # Elk baanvak heeft wel een lijst NWB-wegvakken (NwbId). Staat een daarvan in 'snelweg', dan is het een snelweg.
    snelweg_ids = set(snelweg["WVK_ID"])   # set: Python kan hierin heel snel opzoeken

    def is_snelweg(nwb_ids):
        # "600461021H,508588042H" -> ["600461021", "508588042"]  (lijst maken, spaties en H/T eraf)
        ids = [x.strip().rstrip("HT") for x in nwb_ids.split(",")]
        return any(i in snelweg_ids for i in ids)   # True als minstens één wegvak een snelweg is

    baanvakken = inweva_hr[inweva_hr["NwbId"].apply(is_snelweg)].copy()

    #begin en eind van elk baanvak op volgorde zetten
    # Op de linkerbaan (L) lopen de kilometers af (Hm_van > Hm_naar). between() werkt alleen met (laag, hoog).
    baanvakken["hm_laag"] = baanvakken[["Hm_van", "Hm_naar"]].min(axis=1)
    baanvakken["hm_hoog"] = baanvakken[["Hm_van", "Hm_naar"]].max(axis=1)

    #baanvakken over een kilometersprong eruit
    # Bij een sprong (bijv. A10-ring begint weer bij km 0) is het bereik veel groter dan het baanvak echt lang is,
    # waardoor het ongevallen van elders zou "vangen".
    sprong = (baanvakken["hm_hoog"] - baanvakken["hm_laag"]) > baanvakken["Lengte_baan_meter"] + 1000
    baanvakken = baanvakken[~sprong].copy()   # ~ = niet: houd alles wat geen sprong is

    print(f"{len(baanvakken)} snelwegbaanvakken ({sprong.sum()} over een kilometersprong weggelaten)")
    return baanvakken

In [4]:
baanvakken = maak_baanvakken()   # verwacht: 3209 snelwegbaanvakken (46 weggelaten)
baanvakken

3209 snelwegbaanvakken (46 over een kilometersprong weggelaten)


,VIAS_baannr,Wegnr_van,Ri_van,Hm_van,Baantype_van,Traject_van,Hm_naar,Traject_naar,Lengte_baan_meter,NwbId,etmaal_AL_weekdag_2023,etmaal_AL_weekdag_2022,etmaal_AL_weekdag_2021,etmaal_AL_weekdag_2020,etmaal_AL_weekdag_2019,etmaal_AL_weekdag_2018,hm_laag,hm_hoog
52,174,58.0,L,171370.0,HR,Havendorpweg,170801.0,Ritthem,571,"600401624T,600401625T",16600,16600,15900,15500,18100,17300,170801.0,171370.0
53,175,58.0,R,170826.0,HR,Ritthem,171370.0,Bermweg,544,600094004H,16700,16800,16100,15800,18300,17600,170826.0,171370.0
57,209,58.0,R,170046.0,HR,Ritthem,170826.0,Ritthem,788,600093533H,13800,13900,13200,12800,14600,14300,170046.0,170826.0
58,210,58.0,L,170801.0,HR,Ritthem,169984.0,Ritthem,831,62174078T,14000,14100,13300,12800,14600,14300,169984.0,170801.0
75,291,58.0,L,169984.0,HR,Ritthem,168071.0,Middelburg-Centrum,1898,63175041T,16100,16200,15300,14700,17000,16400,168071.0,169984.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8971,153151,7.0,R,193860.0,HR,Hoogkerk,194358.0,Lijndiensten Bus,569,461559010H,28500,27400,26300,25500,30400,32000,193860.0,194358.0
8996,163234,28.0,L,199801.0,HR,Kp Julianaplein,199604.0,A7 Toekomst,203,600973611T,17100,12100,18500,21100,25200,23500,199604.0,199801.0
9017,163483,20.0,L,19090.0,HR,A20,18212.0,Rijksweg Nr,885,"600991075T,600992375T",30200,26700,28900,28000,30700,30300,18212.0,19090.0
9022,163501,20.0,L,19261.0,HR,A20,19090.0,A20,175,600992372T,29300,27600,27500,26800,30700,30300,19090.0,19261.0


In [5]:
#2. één jaar ongevallen verwerken
# Alles gebruikt de bestanden van hetzelfde jaar (ongevallen, wegvakken, hectopunten),
# zodat de wegvak-ID's altijd bij elkaar passen.

def verwerk_jaar(jaar, baanvakken):
    map_bron = f"data/BRON/01-01-{jaar}_31-12-{jaar}"

    #data inlezen (alleen de kolommen die we nodig hebben)
    # ongevallen: nummer, wegvak, ernst (AP3_CODE), licht (LGD_ID), limiet (MAXSNELHD), type ongeval (AOL_ID)
    ongevallen = pd.read_csv(f"{map_bron}/Ongevallengegevens/ongevallen.txt", encoding="latin-1", dtype=str,
                             usecols=["VKL_NUMMER", "WVK_ID", "AP3_CODE", "LGD_ID", "MAXSNELHD", "AOL_ID"])
    # wegvakken: ID, beheerder, baansoort, routeletter (A/N), wegnummer, richting (R/L)
    wegvakken = pd.read_csv(f"{map_bron}/Netwerkgegevens/wegvakken.txt", encoding="latin-1", dtype=str,
                            usecols=["WVK_ID", "WEGBEHSRT", "BST_CODE", "ROUTELTR", "WEGNUMMER", "RPE_CODE"])

    #relevante rijen selecteren: alleen hoofdrijbanen van snelwegen
    snelweg = wegvakken[
        (wegvakken["WEGBEHSRT"] == "R")      # Rijkswaterstaat
        & (wegvakken["BST_CODE"] == "HR")    # hoofdrijbaan
        & (wegvakken["ROUTELTR"] == "A")     # snelweg
    ]
    snelweg = snelweg.drop(columns=["WEGBEHSRT", "BST_CODE", "ROUTELTR"])  # nu overal hetzelfde

    #ongevallen op de snelwegen
    # merge op WVK_ID plakt weg en richting aan elk ongeval; how="inner" houdt alleen snelwegongevallen
    ongevallen_snelweg = ongevallen.merge(snelweg, on="WVK_ID", how="inner")

    #twee kolommen in zelfde vorm als inweva toevoegen
    ongevallen_snelweg["weg"] = pd.to_numeric(ongevallen_snelweg["WEGNUMMER"])   # "004" -> 4

    #positie van het ongeval = midden van zijn wegvak
    # De kolom HECTOMETER is niet altijd ingevuld (2018: 36% leeg, 2023: 4% leeg), het wegvak (WVK_ID) wel.
    # Een INWEVA-baanvak bestaat uit hele wegvakken, dus het ongeval komt toch bij het juiste baanvak uit
    # (getest: 99,3-99,9% hetzelfde baanvak als met de echte hectometer). Zo werkt het voor elk jaar hetzelfde.
    hectopunten = pd.read_csv(f"{map_bron}/Netwerkgegevens/hectopunten.txt", encoding="latin-1", dtype=str,
                              usecols=["WVK_ID", "HECTOMETER"])
    hectopunten["meter"] = pd.to_numeric(hectopunten["HECTOMETER"]) * 100   # 156 -> 15600
    midden_wegvak = hectopunten.groupby("WVK_ID")["meter"].median()        # middelste paaltje per wegvak
    ongevallen_snelweg["positie_m"] = ongevallen_snelweg["WVK_ID"].map(midden_wegvak)  # opzoeken per ongeval

    #ongevallen koppelen aan baanvakken
    # Regel: een ongeval hoort bij het baanvak op dezelfde weg, in dezelfde richting,
    # waarvan het bereik (hm_laag tot hm_hoog) de positie van het ongeval bevat.
    # 1. merge op weg + richting: elk ongeval tegen ALLE baanvakken van zijn weg en richting
    kandidaten = ongevallen_snelweg.merge(
        baanvakken[["VIAS_baannr", "Wegnr_van", "Ri_van", "hm_laag", "hm_hoog"]],
        left_on=["weg", "RPE_CODE"], right_on=["Wegnr_van", "Ri_van"])
    # 2. alleen de rij houden waar de positie binnen het baanvak valt
    raak = kandidaten[kandidaten["positie_m"].between(kandidaten["hm_laag"], kandidaten["hm_hoog"])]
    # 3. ligt een ongeval precies op de grens van twee baanvakken, dan houden we er één
    gekoppeld = raak.drop_duplicates("VKL_NUMMER").copy()

    gekoppeld["jaar"] = jaar   # zodat we na het samenvoegen weten uit welk jaar elk ongeval komt

    print(f"{jaar}: {len(ongevallen_snelweg)} snelwegongevallen, {len(gekoppeld)} gekoppeld "
          f"({len(gekoppeld) / len(ongevallen_snelweg):.1%})")
    return gekoppeld

In [6]:
#3. alle jaren verwerken en onder elkaar zetten
# [verwerk_jaar(j, baanvakken) for j in JAREN] maakt een lijst met één tabel per jaar;
# pd.concat plakt die onder elkaar tot één tabel. De kolom 'jaar' zegt uit welk jaar elk ongeval komt.
alle_ongevallen = pd.concat([verwerk_jaar(j, baanvakken) for j in JAREN], ignore_index=True)

# Resultaat: één rij per gekoppeld snelwegongeval, met het baanvak (VIAS_baannr) waar het gebeurde
alle_ongevallen

2018: 27219 snelwegongevallen, 25877 gekoppeld (95.1%)
2019: 36266 snelwegongevallen, 35008 gekoppeld (96.5%)
2020: 20663 snelwegongevallen, 20051 gekoppeld (97.0%)
2021: 21768 snelwegongevallen, 21053 gekoppeld (96.7%)
2022: 24210 snelwegongevallen, 23494 gekoppeld (97.0%)
2023: 29982 snelwegongevallen, 29115 gekoppeld (97.1%)
2024: 27887 snelwegongevallen, 27093 gekoppeld (97.2%)
2025: 27808 snelwegongevallen, 27030 gekoppeld (97.2%)


,VKL_NUMMER,AP3_CODE,AOL_ID,MAXSNELHD,LGD_ID,WVK_ID,WEGNUMMER,RPE_CODE,weg,positie_m,VIAS_baannr,Wegnr_van,Ri_van,hm_laag,hm_hoog,jaar
0,20180004322,LET,8,120,1,335159010,067,R,67.0,30700.0,9655,67.0,R,28079.0,37038.0,2018
1,20180004328,LET,8,120,1,338141017,002,L,2.0,182300.0,9766,2.0,L,181248.0,182494.0,2018
2,20180004345,LET,0,130,1,312182050,002,L,2.0,140800.0,8788,2.0,L,140390.0,141469.0,2018
3,20180004892,LET,8,120,2,205288009,020,L,20.0,44500.0,3594,20.0,L,43205.0,45971.0,2018
4,20180004909,LET,4,130,2,476560006,007,R,7.0,206300.0,15118,7.0,R,205362.0,209723.0,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208716,20250176965,UMS,8,100,3,600106146,004,R,4.0,237750.0,1617,4.0,R,236651.0,241022.0,2025
208717,20250176975,UMS,7,100,1,600280907,004,R,4.0,6500.0,4728,4.0,R,6119.0,7764.0,2025
208718,20250176987,UMS,8,100,3,225363001,004,L,4.0,5650.0,4770,4.0,L,5153.0,6128.0,2025
208719,20250176995,LET,8,100,2,414271007,012,R,12.0,148850.0,13411,12.0,R,148365.0,149300.0,2025


In [7]:
#controle: in welk baanvak zit een ongeval?
# Ongeval 20230062249 -> baanvak 11553 (A2 rechts, km 222,15-224,19, tussen A73 en Echt)
voorbeeld = alle_ongevallen[alle_ongevallen["VKL_NUMMER"] == "20230062249"]
voorbeeld[["VKL_NUMMER", "jaar", "WEGNUMMER", "RPE_CODE", "positie_m", "VIAS_baannr", "hm_laag", "hm_hoog"]]

,VKL_NUMMER,jaar,WEGNUMMER,RPE_CODE,positie_m,VIAS_baannr,hm_laag,hm_hoog
125484,20230062249,2023,002,R,223150.0,11553,222154.0,224191.0


In [9]:
#aantal ongevallen per baanvak per jaar
# groupby op baanvak én jaar, size() telt het aantal rijen (= ongevallen) per combinatie.
# unstack zet de jaren naast elkaar, zodat je per baanvak de jaren kunt vergelijken.
per_baanvak_jaar = alle_ongevallen.groupby(["VIAS_baannr", "jaar"]).size().unstack(fill_value=0)

# voorbeeld: baanvak 3901 (A4 rechts bij Hoofddorp) -> 2018: 27, 2019: 19, 2020: 6, ... 2023: 18
per_baanvak_jaar

jaar,2018,2019,2020,2021,2022,2023,2024,2025
VIAS_baannr,,,,,,,,
174,0,4,3,3,1,5,3,4
175,6,6,4,3,2,5,4,7
209,0,1,1,4,4,2,0,2
210,1,0,2,4,0,2,2,1
291,1,1,3,2,4,2,2,2
...,...,...,...,...,...,...,...,...
144823,0,6,0,6,3,2,3,1
153151,2,5,2,2,5,3,0,0
163234,0,1,3,5,1,2,3,0


In [12]:
#verkeer per baanvak per jaar
# In INWEVA staat de drukte in zes kolommen naast elkaar (één per jaar).
# melt zet die onder elkaar: per baanvak zes rijen, met een kolom 'jaar' en een kolom 'intensiteit'.
jaarkolommen = [f"etmaal_AL_weekdag_{j}" for j in range(2018, 2024)]
verkeer = baanvakken.melt(id_vars=["VIAS_baannr", "Lengte_baan_meter"], value_vars=jaarkolommen,
                          var_name="kolom", value_name="intensiteit")
verkeer["jaar"] = verkeer["kolom"].str[-4:].astype(int)   # "etmaal_AL_weekdag_2018" -> 2018

#ongevallen per baanvak per jaar (lange vorm, zodat het past bij 'verkeer')
telling = alle_ongevallen.groupby(["VIAS_baannr", "jaar"]).size().rename("ongevallen").reset_index()

#samenvoegen: verkeer links, zodat ook baanvakken zonder ongevallen meetellen (die krijgen 0)
resultaat = verkeer.merge(telling, on=["VIAS_baannr", "jaar"], how="left")
resultaat["ongevallen"] = resultaat["ongevallen"].fillna(0)

#voertuigkilometers en risico
# vkm = voertuigen per dag x 365 dagen x lengte in km; in miljoenen
resultaat["vkm_mln"] = resultaat["intensiteit"] * 365 * resultaat["Lengte_baan_meter"] / 1000 / 1_000_000
resultaat["risico"] = resultaat["ongevallen"] / resultaat["vkm_mln"] * 100   # per 100 miljoen vkm

resultaat[resultaat["VIAS_baannr"] == 3901]

,VIAS_baannr,Lengte_baan_meter,kolom,intensiteit,jaar,ongevallen,vkm_mln,risico
596,3901,1388,etmaal_AL_weekdag_2018,97600,2018,27.0,49.446112,54.604900
3805,3901,1388,etmaal_AL_weekdag_2019,99300,2019,19.0,50.307366,37.767829
7014,3901,1388,etmaal_AL_weekdag_2020,76900,2020,6.0,38.959078,15.400775
10223,3901,1388,etmaal_AL_weekdag_2021,83100,2021,13.0,42.100122,30.878770
13432,3901,1388,etmaal_AL_weekdag_2022,92700,2022,10.0,46.963674,21.293053
16641,3901,1388,etmaal_AL_weekdag_2023,97100,2023,18.0,49.192802,36.590719
